# 🚆 Tren Sistemi Tanılama Notebook'u

Bu notebook şu soruları yanıtlar:
1. **DB REST API (Almanya)** gerçekten veri döndürüyor mu?
2. **Trenitalia / Playwright (İtalya)** çalışıyor mu?
3. **Veritabanları** (istasyon önbellekleri + search_cache) ne durumda?
4. `train_finder.get_trips()` end-to-end çalışıyor mu?
5. Sonuçlar `train_cache` tablosuna yazıldı mı?

Her hücre bağımsız çalışacak şekilde tasarlandı — hataları burada görürsünüz.

---
## 0. Kurulum & Yol Ayarı

In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path('__file__').resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Python:', sys.version)

for p in ['backend', 'data']:
    full = PROJECT_ROOT / p
    status = 'VAR OK' if full.exists() else 'YOK HATA'
    print(f'  {p}/  -> {status}')


In [2]:
# Temel kütüphaneler
import sqlite3
import json
import requests
import pandas as pd
from datetime import datetime, date, timedelta
from io import StringIO

# Test tarihi: bugünden 15 gün sonrası (cache geçmiş tarihleri reddeder)
TEST_DATE = (date.today() + timedelta(days=15)).isoformat()
print("Test tarihi:", TEST_DATE)

Test tarihi: 2026-05-20


---
## 1. Veritabanı Durumu — Genel Bakış

In [3]:
DATA_DIR = PROJECT_ROOT / "data"

DB_FILES = {
    "search_cache.db":         "Tren/Otobüs/Uçuş sonuç önbelleği",
    "deutschland_stationen.db": "Alman DB istasyon ID önbelleği",
    "italya_istasyonlar.db":    "İtalyan Trenitalia istasyon tablosu",
    "flixbus_europe.db":        "FlixBus durak veritabanı",
}

print(f"{'Dosya':<35} {'Boyut':>8}  Durum")
print("-" * 60)
for fname, desc in DB_FILES.items():
    fp = DATA_DIR / fname
    if fp.exists():
        size_kb = fp.stat().st_size / 1024
        print(f"{fname:<35} {size_kb:>6.1f} KB  ✅  {desc}")
    else:
        print(f"{fname:<35} {'---':>8}  ❌  BULUNAMADI")

Dosya                                  Boyut  Durum
------------------------------------------------------------
search_cache.db                      612.0 KB  ✅  Tren/Otobüs/Uçuş sonuç önbelleği
deutschland_stationen.db              12.0 KB  ✅  Alman DB istasyon ID önbelleği
italya_istasyonlar.db                168.0 KB  ✅  İtalyan Trenitalia istasyon tablosu
flixbus_europe.db                     12.0 KB  ✅  FlixBus durak veritabanı


---
## 2. `search_cache.db` — Tablo ve Satır Sayıları

In [4]:
CACHE_DB = DATA_DIR / "search_cache.db"

with sqlite3.connect(CACHE_DB) as conn:
    tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()

print("Tablolar:")
for (tbl,) in tables:
    cnt = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<20} → {cnt} satır")

Tablolar:
  bus_cache            → 32 satır
  flight_cache         → 43 satır
  train_cache          → 2 satır


In [5]:
# train_cache tablosunun son 10 kaydı
with sqlite3.connect(CACHE_DB) as conn:
    rows = conn.execute(
        "SELECT from_city, to_city, date, cached_at, length(payload) AS payload_bytes "
        "FROM train_cache ORDER BY cached_at DESC LIMIT 10"
    ).fetchall()

if rows:
    df_cache = pd.DataFrame(rows, columns=["from_city", "to_city", "date", "cached_at", "payload_bytes"])
    display(df_cache)
else:
    print("⚠️  train_cache tablosu BOŞ — henüz hiç tren araması önbelleklenmemiş.")

,from_city,to_city,date,cached_at,payload_bytes
0,stuttgart,nurnberg,2026-05-05,2026-05-04T16:36:56.158063,1629
1,munich,nurnberg,2026-05-05,2026-05-04T16:36:56.082870,1726


---
## 3. `deutschland_stationen.db` — Alman İstasyon Önbelleği

In [6]:
DE_DB = DATA_DIR / "deutschland_stationen.db"

with sqlite3.connect(DE_DB) as conn:
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    print("Tablolar:", [t[0] for t in tables])

    if tables:
        rows = conn.execute("SELECT * FROM stationen ORDER BY query").fetchall()
        if rows:
            df_de = pd.DataFrame(rows, columns=["query", "station_id", "name"])
            print(f"\n{len(df_de)} istasyon önbelleğe alınmış:")
            display(df_de)
        else:
            print("⚠️  stationen tablosu BOŞ — henüz hiç Alman istasyon araması yapılmamış.")

Tablolar: ['stationen']

3 istasyon önbelleğe alınmış:


,query,station_id,name
0,munich,8096013,MUNICH (MÜNCHEN)
1,nurnberg,8096025,NÜRNBERG
2,stuttgart,8000096,Stuttgart Hbf


---
## 4. `italya_istasyonlar.db` — İtalyan İstasyon Tablosu

In [7]:
IT_DB = DATA_DIR / "italya_istasyonlar.db"

with sqlite3.connect(IT_DB) as conn:
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    print("Tablolar:", [t[0] for t in tables])

    total = conn.execute("SELECT COUNT(*) FROM istasyonlar").fetchone()[0]
    print(f"Toplam istasyon sayısı: {total}")

    print("\nÖrnek 10 kayıt:")
    rows = conn.execute("SELECT id, isim, search_terms FROM istasyonlar LIMIT 10").fetchall()
    df_it = pd.DataFrame(rows, columns=["id", "isim", "search_terms"])
    display(df_it)

Tablolar: ['istasyonlar', 'otobus_duraklari']
Toplam istasyon sayısı: 206

Örnek 10 kayıt:


,id,isim,search_terms
0,830000060,Torino Rebaudengo Fossata,None
1,830000088,Torino Aeroporto Di Caselle,None
2,830000097,Torino Grosseto,None
3,830000109,Romagnano Sesia,None
4,830000219,Torino Porta Nuova,None
5,830000222,Torino Porta Susa,None
6,830000228,Torino Stura,None
7,830000452,Torino Lingotto,None
8,830000996,Torino ( Tutte Le Stazioni ),None
9,830001326,Milano Greco Pirelli,None


In [8]:
# Büyük şehirlerin ana istasyonlarının veritabanında olup olmadığını kontrol et
MAJOR_CITIES = [
    ("roma termini",              "Roma Termini"),
    ("milano centrale",           "Milano Centrale"),
    ("venezia s. lucia",          "Venezia S. Lucia"),
    ("firenze s. m. novella",     "Firenze S. M. Novella"),
    ("bologna centrale",          "Bologna Centrale"),
    ("napoli centrale",           "Napoli Centrale"),
    ("torino porta nuova",        "Torino Porta Nuova"),
]

print(f"{'İstasyon Adı':<35} {'ID':>8}  Durum")
print("-" * 55)
with sqlite3.connect(IT_DB) as conn:
    for key, display_name in MAJOR_CITIES:
        row = conn.execute(
            "SELECT id FROM istasyonlar WHERE LOWER(isim)=?", (key,)
        ).fetchone()
        if row:
            print(f"{display_name:<35} {row[0]:>8}  ✅")
        else:
            print(f"{display_name:<35} {'---':>8}  ❌ VERİTABANINDA YOK")

İstasyon Adı                              ID  Durum
-------------------------------------------------------
Roma Termini                        830008409  ✅
Milano Centrale                     830001700  ✅
Venezia S. Lucia                    830002593  ✅
Firenze S. M. Novella               830006421  ✅
Bologna Centrale                    830005043  ✅
Napoli Centrale                     830009218  ✅
Torino Porta Nuova                  830000219  ✅


---
## 5. Ülke Tespiti — `_detect_country` Testi

In [9]:
# Backend modülünü import et
from backend.train_finder import _detect_country, _norm, _DE_CITIES, _IT_CITIES

print(f"DE şehir seti boyutu: {len(_DE_CITIES)}")
print(f"IT şehir seti boyutu: {len(_IT_CITIES)}")

test_pairs = [
    ("Berlin",    "Munich",   "DE bekleniyor"),
    ("Hamburg",   "Cologne",  "DE bekleniyor"),
    ("Frankfurt", "Stuttgart","DE bekleniyor"),
    ("Rome",      "Milan",    "IT bekleniyor"),
    ("Venice",    "Florence", "IT bekleniyor"),
    ("Berlin",    "Rome",     "cross bekleniyor"),
    ("Paris",     "Lyon",     "unknown bekleniyor"),
]

print(f"\n{'Güzergah':<30} {'Tespit':<10}  Not")
print("-" * 60)
for orig, dest, note in test_pairs:
    result = _detect_country(orig, dest)
    ok = "✅" if result.lower().split()[0] in note.lower() else "❌"
    print(f"{orig} → {dest:<20} {result:<10}  {ok}  {note}")

DE şehir seti boyutu: 62
IT şehir seti boyutu: 63

Güzergah                       Tespit      Not
------------------------------------------------------------
Berlin → Munich               DE          ✅  DE bekleniyor
Hamburg → Cologne              DE          ✅  DE bekleniyor
Frankfurt → Stuttgart            DE          ✅  DE bekleniyor
Rome → Milan                IT          ✅  IT bekleniyor
Venice → Florence             IT          ✅  IT bekleniyor
Berlin → Rome                 cross       ✅  cross bekleniyor
Paris → Lyon                 unknown     ✅  unknown bekleniyor


---
## 6. DB REST API — Doğrudan HTTP Testi (Almanya)

In [10]:
# 6a. İstasyon arama (location lookup)
DB_REST = "https://v6.db.transport.rest"

print("=== DB REST API — /locations testi ===")
for city in ["Berlin", "München", "Hamburg"]:
    try:
        r = requests.get(
            f"{DB_REST}/locations",
            params={"query": city, "results": 3, "stops": "true"},
            timeout=10,
        )
        r.raise_for_status()
        data = r.json()
        stops = [s for s in data if s.get("type") in ("stop", "station")]
        if stops:
            best = stops[0]
            print(f"  ✅ {city:<15} → id={best['id']}  name={best.get('name')}")
        else:
            print(f"  ⚠️  {city:<15} → sonuç yok (HTTP {r.status_code})")
    except Exception as e:
        print(f"  ❌ {city:<15} → HATA: {e}")

=== DB REST API — /locations testi ===
  ✅ Berlin          → id=8011160  name=Berlin Hbf
  ✅ München         → id=8096013  name=MÜNCHEN
  ✅ Hamburg         → id=8096009  name=HAMBURG


In [11]:
# 6b. Sefer arama (journeys)
# Önce Berlin ve München istasyon ID'lerini al
def get_station_id(city: str) -> str | None:
    try:
        r = requests.get(
            f"{DB_REST}/locations",
            params={"query": city, "results": 3, "stops": "true"},
            timeout=10,
        )
        r.raise_for_status()
        stops = [s for s in r.json() if s.get("type") in ("stop", "station")]
        return str(stops[0]["id"]) if stops else None
    except Exception as e:
        print(f"  ❌ İstasyon araması başarısız ({city}): {e}")
        return None

berlin_id  = get_station_id("Berlin")
münchen_id = get_station_id("München")
print(f"Berlin  ID: {berlin_id}")
print(f"München ID: {münchen_id}")

Berlin  ID: 8011160
München ID: 8096013


In [12]:
# 6c. /journeys çağrısı
if berlin_id and münchen_id:
    print(f"Berlin → München seferleri ({TEST_DATE})")
    try:
        r = requests.get(
            f"{DB_REST}/journeys",
            params={
                "from":            berlin_id,
                "to":              münchen_id,
                "departure":       f"{TEST_DATE}T06:00:00",
                "results":         5,
                "tickets":         "true",
                "nationalExpress": "true",
                "national":        "true",
                "regionalExp":     "true",
                "regional":        "true",
                "suburban":        "false",
                "subway":          "false",
                "tram":            "false",
                "bus":             "false",
                "ferry":           "false",
                "taxi":            "false",
            },
            timeout=20,
        )
        r.raise_for_status()
        journeys = r.json().get("journeys", [])
        print(f"  HTTP {r.status_code} — {len(journeys)} sefer bulundu")

        for j in journeys[:3]:
            legs = j.get("legs", [])
            dep  = legs[0].get("departure", "?") if legs else "?"
            arr  = legs[-1].get("arrival", "?") if legs else "?"
            price_obj = j.get("price")
            price = price_obj.get("amount", "?") if price_obj else "fiyat yok"
            train_legs = [l for l in legs if l.get("line")]
            aktarma = max(0, len(train_legs) - 1)
            print(f"  Kalkış: {dep[:16]}  Varış: {arr[:16]}  Fiyat: {price} EUR  Aktarma: {aktarma}")

    except Exception as e:
        print(f"  ❌ /journeys çağrısı başarısız: {e}")
else:
    print("⚠️  İstasyon ID'leri alınamadı, sefer araması atlandı.")

Berlin → München seferleri (2026-05-20)
  HTTP 200 — 5 sefer bulundu
  Kalkış: 2026-05-20T08:29  Varış: 2026-05-20T13:08  Fiyat: 79.99 EUR  Aktarma: 0
  Kalkış: 2026-05-20T08:36  Varış: 2026-05-20T12:46  Fiyat: 74.99 EUR  Aktarma: 0
  Kalkış: 2026-05-20T09:36  Varış: 2026-05-20T13:43  Fiyat: 74.99 EUR  Aktarma: 0


---
## 7. `train_finder._resolve_station_de` — İstasyon Çözümleme + Önbellekleme

In [13]:
from backend.train_finder import _resolve_station_de

test_cities_de = ["Berlin", "Munich", "Hamburg", "Frankfurt", "Cologne"]

print(f"{'Şehir':<15} {'İstasyon ID':<15} {'Resmi Ad'}")
print("-" * 55)
for city in test_cities_de:
    result = _resolve_station_de(city)
    if result:
        sid, name = result
        print(f"{city:<15} {sid:<15} {name}")
    else:
        print(f"{city:<15} ❌ ÇÖZÜMLENEMEDI")

Şehir           İstasyon ID     Resmi Ad
-------------------------------------------------------
Berlin          8011160         Berlin Hbf
Munich          8096013         MUNICH (MÜNCHEN)
Hamburg         8096009         HAMBURG
Frankfurt       8096021         FRANKFURT(MAIN)
Cologne         8096022         COLOGNE


In [14]:
# İstasyon önbelleğini tekrar kontrol et (yukarıdaki çağrılar DB'ye yazmalıydı)
with sqlite3.connect(DE_DB) as conn:
    rows = conn.execute("SELECT query, station_id, name FROM stationen ORDER BY query").fetchall()

df_de_after = pd.DataFrame(rows, columns=["query", "station_id", "name"])
print(f"Alman istasyon önbelleği: {len(df_de_after)} kayıt")
display(df_de_after)

Alman istasyon önbelleği: 7 kayıt


,query,station_id,name
0,berlin,8011160,Berlin Hbf
1,cologne,8096022,COLOGNE
2,frankfurt,8096021,FRANKFURT(MAIN)
3,hamburg,8096009,HAMBURG
4,munich,8096013,MUNICH (MÜNCHEN)
5,nurnberg,8096025,NÜRNBERG
6,stuttgart,8000096,Stuttgart Hbf


---
## 8. `_get_trips_de` — Almanya Tren Araması (Modül Üzerinden)

In [15]:
from backend.train_finder import _get_trips_de

print(f"Berlin → München  ({TEST_DATE})")
df_de_trips = _get_trips_de("Berlin", "Munich", TEST_DATE)

print(f"\nSonuç: {len(df_de_trips)} sefer")
if not df_de_trips.empty:
    display(df_de_trips[["origin","destination","departure_dt","arrival_dt","duration_min","price_eur","stops","provider"]])
else:
    print("⚠️  DataFrame BOŞ — API çağrısı başarısız ya da sonuç yok.")

Berlin → München  (2026-05-20)
⚠️  DB REST journeys failed Berlin Hbf→MUNICH (MÜNCHEN): 503 Server Error: Service Unavailable for url: https://v6.db.transport.rest/journeys?from=8011160&to=8096013&departure=2026-05-20T06%3A00%3A00&results=15&tickets=true&nationalExpress=true&national=true&regionalExp=true&regional=true&suburban=false&subway=false&tram=false&bus=false&ferry=false&taxi=false

Sonuç: 0 sefer
⚠️  DataFrame BOŞ — API çağrısı başarısız ya da sonuç yok.


---
## 9. Trenitalia BFF API — Doğrudan HTTP Testi (İtalya)

In [16]:
# 9a. Trenitalia istasyon arama (Playwright olmadan doğrudan HTTP)
TRENITALIA_BFF = "https://www.lefrecce.it/Channels.Website.BFF.WEB/website"

print("=== Trenitalia BFF — /locations/search testi ===")
for city in ["Roma", "Milano", "Venezia", "Firenze", "Napoli"]:
    try:
        r = requests.get(
            f"{TRENITALIA_BFF}/locations/search",
            params={"name": city, "limit": 3},
            headers={
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0.0.0",
                "Accept":     "application/json",
                "Referer":    "https://www.lefrecce.it/",
            },
            timeout=10,
        )
        if r.status_code == 200:
            data = r.json()
            if data:
                best = data[0]
                print(f"  ✅ {city:<12} → id={best.get('id')}  name={best.get('name')}")
            else:
                print(f"  ⚠️  {city:<12} → boş sonuç (HTTP 200)")
        else:
            print(f"  ⚠️  {city:<12} → HTTP {r.status_code} (muhtemelen CORS/bot engeli)")
    except Exception as e:
        print(f"  ❌ {city:<12} → HATA: {e}")

=== Trenitalia BFF — /locations/search testi ===
  ✅ Roma         → id=830008349  name=Roma ( Tutte Le Stazioni )
  ✅ Milano       → id=830001650  name=Milano ( Tutte Le Stazioni )
  ✅ Venezia      → id=830002998  name=Venezia ( Tutte Le Stazioni )
  ✅ Firenze      → id=830006998  name=Firenze ( Tutte Le Stazioni )
  ✅ Napoli       → id=830009993  name=Napoli ( Tutte Le Stazioni )


In [27]:
# 9b. Trenitalia ticket/solutions — doğrudan POST testi
# (Playwright olmadan — bot koruması varsa bu hücre başarısız olabilir)
ROMA_ID    = 830008409  # Roma Termini (italya_istasyonlar.db'den)
MILANO_ID  = 830001700  # Milano Centrale

body = {
    "departureLocationId": ROMA_ID,
    "arrivalLocationId":   MILANO_ID,
    "departureTime":       f"{TEST_DATE}T08:00:00.000+01:00",
    "adults":  1,
    "children": 0,
    "criteria": {
        "frecceOnly":   False,
        "regionalOnly": False,
        "noChanges":    False,
        "order":        "DEPARTURE_DATE",
        "limit":        5,
        "offset":       0,
    },
    "advancedSearchRequest": {"bestFare": False},
}

print(f"Trenitalia BFF /ticket/solutions — Roma→Milano ({TEST_DATE})")
try:
    r = requests.post(
        f"{TRENITALIA_BFF}/ticket/solutions",
        json=body,
        headers={
            "User-Agent":   "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0.0.0",
            "Accept":       "application/json",
            "Content-Type": "application/json",
            "Referer":      "https://www.lefrecce.it/",
        },
        timeout=20,
    )
    print(f"  HTTP {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        solutions = data.get("solutions", [])
        print(f"  {len(solutions)} çözüm bulundu")
        for sol in solutions[:3]:
            inner = sol.get("solution", sol)
            price = inner.get("price", {}).get("amount", "?")
            dep   = inner.get("departureTime", "?")[:16]
            arr   = inner.get("arrivalTime",   "?")[:16]
            print(f"    Kalkış: {dep}  Varış: {arr}  Fiyat: {price} EUR")
    else:
        print(f"  Yanıt metni: {r.text[:500]}")
except Exception as e:
    print(f"  ❌ HATA: {e}")
    print("  NOT: Playwright bot korumasını geçmek için gerekli. Doğrudan HTTP çağrısı engellenmiş olabilir.")

Trenitalia BFF /ticket/solutions — Roma→Milano (2026-05-20)
  HTTP 200
  5 çözüm bulundu
    Kalkış: 2026-05-20T08:10  Varış: 2026-05-20T11:50  Fiyat: 64.9 EUR
    Kalkış: 2026-05-20T08:20  Varış: 2026-05-20T11:35  Fiyat: 50.9 EUR
    Kalkış: 2026-05-20T08:50  Varış: 2026-05-20T11:58  Fiyat: 50.9 EUR


---
## 10. Playwright Varlık Kontrolü

In [19]:
# Playwright sync API Jupyter'de Windows asyncio loop ile uyumsuz.
# Cozum: Playwright'i ayri bir subprocess'te calistir.
import subprocess, sys

try:
    import playwright  # kurulu mu kontrol
    print('Playwright modulu: OK (kurulu)')
except ImportError:
    print('HATA playwright KURULU DEGIL')
    print('   Cozum: uv add playwright && uv run playwright install chromium')
    raise SystemExit

# Gercek Chromium testini subprocess ile yap
code = '''
from playwright.sync_api import sync_playwright
with sync_playwright() as pw:
    b = pw.chromium.launch(headless=True)
    print('Chromium calisiyor -- versiyon:', b.version)
    b.close()
'''
r = subprocess.run(['uv', 'run', 'python', '-c', code],
                   capture_output=True, text=True, timeout=30)
if r.returncode == 0:
    print(r.stdout.strip())
else:
    print('UYARI Chromium testi basarisiz:')
    print(r.stderr[:300])
    print('   Cozum: uv run playwright install chromium')


---
## 11. `_get_trips_it` — İtalya Tren Araması (Playwright ile)

In [21]:
import subprocess

print(f'Rome -> Milan ({TEST_DATE}) -- subprocess ile Trenitalia...')

# PROJECT_ROOT mutlak yolunu subprocess'e gecir
code = (
    f'import sys; sys.path.insert(0, r"{PROJECT_ROOT}"); '
    'from backend.train_finder import _get_trips_it; '
    f'df = _get_trips_it("Rome", "Milan", "{TEST_DATE}"); '
    'print("EMPTY") if df.empty else '
    '(print(f"ROWS={len(df)}"), '
    'print(df[["departure_dt","arrival_dt","duration_min","price_eur","stops","provider"]].head(5).to_string(index=False)))'
)

r = subprocess.run(['uv', 'run', 'python', '-c', code],
                   capture_output=True, text=True, timeout=120,
                   cwd=str(PROJECT_ROOT))

if r.returncode != 0 or not r.stdout.strip():
    print('HATA:')
    print(r.stderr[:500] if r.stderr else '(cikti yok)')
    df_it_trips = pd.DataFrame()
elif r.stdout.strip().startswith('EMPTY'):
    print('DataFrame BOS. Olasi nedenler:')
    print('  1. playwright veya Chromium kurulu degil')
    print('  2. italya_istasyonlar.db de Roma/Milano istasyon ID si eksik')
    print('  3. Trenitalia bot engellemesi')
    print('  4. Ag baglantisi yok')
    df_it_trips = pd.DataFrame()
else:
    print(r.stdout.strip())
    df_it_trips = pd.DataFrame({'info': ['Subprocess ciktisi yukaridaki tabloda']})


---
## 12. `train_finder.get_trips()` — Tam End-to-End Test

In [22]:
from backend.train_finder import get_trips

test_routes = [
    ("Berlin",  "Munich",  TEST_DATE, "DE"),
    ("Hamburg", "Cologne", TEST_DATE, "DE"),
]

results = {}
for orig, dest, dt, label in test_routes:
    print(f"\n{'='*55}")
    print(f"  {orig} → {dest}  ({label})  [{dt}]")
    print(f"{'='*55}")
    df = get_trips(orig, dest, dt)
    results[(orig, dest)] = df
    if not df.empty:
        print(f"\n✅ {len(df)} sefer bulundu:")
        display(df[["departure_dt","arrival_dt","duration_min","price_eur","stops","provider"]].head(5))
    else:
        print("⚠️  Sonuç yok.")


  Berlin → Munich  (DE)  [2026-05-20]
🚆 Train search: 'Berlin' → 'Munich'  [DE]  date=2026-05-20
✅ DE train search: 5 results Berlin Hbf→MUNICH (MÜNCHEN)
💾 train cache SAVE berlin→munich 2026-05-20 (5 rows)

✅ 5 sefer bulundu:


,departure_dt,arrival_dt,duration_min,price_eur,stops,provider
0,2026-05-20 08:36:00,2026-05-20 12:46:00,250,74.99,0,DB
1,2026-05-20 09:36:00,2026-05-20 13:43:00,247,74.99,0,DB
2,2026-05-20 10:36:00,2026-05-20 14:45:00,249,75.99,0,DB
3,2026-05-20 08:29:00,2026-05-20 13:08:00,279,79.99,0,DB
4,2026-05-20 10:29:00,2026-05-20 15:08:00,279,80.99,0,DB



  Hamburg → Cologne  (DE)  [2026-05-20]
🚆 Train search: 'Hamburg' → 'Cologne'  [DE]  date=2026-05-20
✅ DE train search: 5 results HAMBURG→COLOGNE
💾 train cache SAVE hamburg→cologne 2026-05-20 (5 rows)

✅ 5 sefer bulundu:


,departure_dt,arrival_dt,duration_min,price_eur,stops,provider
0,2026-05-20 09:23:00,2026-05-20 13:48:00,265,43.99,0,DB
1,2026-05-20 08:23:00,2026-05-20 12:48:00,265,55.99,0,DB
2,2026-05-20 10:23:00,2026-05-20 14:48:00,265,55.99,0,DB
3,2026-05-20 11:23:00,2026-05-20 15:48:00,265,55.99,0,DB
4,2026-05-20 12:23:00,2026-05-20 16:48:00,265,55.99,0,DB


---
## 13. Önbellek Kontrolü — get_trips() Sonrası

In [23]:
# train_cache tablosunda yeni kayıtlar oluştu mu?
with sqlite3.connect(CACHE_DB) as conn:
    rows = conn.execute(
        "SELECT from_city, to_city, date, cached_at, length(payload) AS bytes "
        "FROM train_cache ORDER BY cached_at DESC LIMIT 20"
    ).fetchall()

if rows:
    df_cache_after = pd.DataFrame(rows, columns=["from_city", "to_city", "date", "cached_at", "bytes"])
    print(f"train_cache: {len(df_cache_after)} kayıt")
    display(df_cache_after)
else:
    print("⚠️  train_cache hâlâ BOŞ — get_trips() boş DataFrame döndürdüğü için önbelleğe yazılmadı.")

train_cache: 4 kayıt


,from_city,to_city,date,cached_at,bytes
0,hamburg,cologne,2026-05-20,2026-05-05T06:29:38.890052,1501
1,berlin,munich,2026-05-20,2026-05-05T06:29:37.864954,1711
2,stuttgart,nurnberg,2026-05-05,2026-05-04T16:36:56.158063,1629
3,munich,nurnberg,2026-05-05,2026-05-04T16:36:56.082870,1726


In [24]:
# Önbellekten bir kaydı okuyup doğrula
with sqlite3.connect(CACHE_DB) as conn:
    row = conn.execute(
        "SELECT from_city, to_city, date, payload, cached_at FROM train_cache LIMIT 1"
    ).fetchone()

if row:
    from_city, to_city, date_val, payload, cached_at = row
    print(f"Önbellekteki kayıt: {from_city} → {to_city}  [{date_val}]  cached_at={cached_at}")
    
    df_from_cache = pd.read_json(StringIO(payload), orient="records")
    # Tarih sütunlarını dönüştür
    for col in ["departure_dt", "arrival_dt"]:
        if col in df_from_cache.columns:
            df_from_cache[col] = pd.to_datetime(df_from_cache[col], errors="coerce")
    
    print(f"Payload'dan okunan DataFrame: {len(df_from_cache)} satır")
    display(df_from_cache.head(3))
else:
    print("⚠️  Önbellekte gösterilecek kayıt yok.")

Önbellekteki kayıt: munich → nurnberg  [2026-05-05]  cached_at=2026-05-04T16:36:56.082870
Payload'dan okunan DataFrame: 5 satır


,origin,destination,date,departure_dt,arrival_dt,duration_min,price_eur,url,provider,stops
0,MUNICH (MÜNCHEN),NÜRNBERG,2026-05-05,2026-05-05 08:49:00,2026-05-05 10:02:00,73,35.99,https://www.bahn.de/buchung/start#sts=true&so=...,DB,0
1,MUNICH (MÜNCHEN),NÜRNBERG,2026-05-05,2026-05-05 08:45:00,2026-05-05 09:57:00,72,35.99,https://www.bahn.de/buchung/start#sts=true&so=...,DB,0
2,MUNICH (MÜNCHEN),NÜRNBERG,2026-05-05,2026-05-05 09:12:00,2026-05-05 10:23:00,71,35.99,https://www.bahn.de/buchung/start#sts=true&so=...,DB,0


---
## 14. Önbellek — İkinci Çağrı (HIT Testi)

In [25]:
# Aynı güzergahı tekrar çağır — bu sefer önbellekten gelmeli
import time

print("Önbellek HIT testi: Berlin → Munich")
t0 = time.time()
df_cached = get_trips("Berlin", "Munich", TEST_DATE)
elapsed = time.time() - t0

if not df_cached.empty:
    print(f"✅ {len(df_cached)} sefer döndü  —  süre: {elapsed:.2f}s")
    if elapsed < 1.0:
        print("   (Hızlı yanıt → önbellekten geldi 💾)")
    else:
        print(f"   (Yavaş yanıt ({elapsed:.1f}s) → API'den geldi, önbellek MISS)")
else:
    print("⚠️  Önbellekten de sonuç gelmedi — orijinal arama boş döndü.")

Önbellek HIT testi: Berlin → Munich
💾 train cache HIT  berlin→munich 2026-05-20
✅ 5 sefer döndü  —  süre: 0.01s
   (Hızlı yanıt → önbellekten geldi 💾)


---
## 15. Özet & Tanı Raporu

In [26]:
print("=" * 60)
print("          TREN SİSTEMİ TANILAMA RAPORU")
print("=" * 60)

# 1. DB REST API erişilebilirliği
try:
    r = requests.get(f"{DB_REST}/locations", params={"query": "Berlin", "results": 1}, timeout=5)
    db_api_ok = r.status_code == 200
except Exception:
    db_api_ok = False
print(f"  DB REST API (Almanya):         {'✅ Erişilebilir' if db_api_ok else '❌ ERİŞİLEMİYOR'}")

# 2. Playwright
try:
    from playwright.sync_api import sync_playwright
    playwright_ok = True
except ImportError:
    playwright_ok = False
print(f"  Playwright (İtalya için):       {'✅ Yüklü' if playwright_ok else '❌ KURULU DEĞİL'}")

# 3. İtalyan istasyon DB
with sqlite3.connect(IT_DB) as conn:
    it_count = conn.execute("SELECT COUNT(*) FROM istasyonlar").fetchone()[0]
    roma_ok  = conn.execute("SELECT id FROM istasyonlar WHERE LOWER(isim)='roma termini' LIMIT 1").fetchone()
print(f"  İtalyan istasyon DB:           {'✅' if it_count > 0 else '❌'} ({it_count} istasyon)")
print(f"  Roma Termini istasyon kaydı:   {'✅ Mevcut' if roma_ok else '❌ EKSİK (Trenitalia araması başarısız olur)'}")

# 4. Alman istasyon DB
with sqlite3.connect(DE_DB) as conn:
    de_count = conn.execute("SELECT COUNT(*) FROM stationen").fetchone()[0]
print(f"  Alman istasyon önbelleği:      {'✅' if de_count > 0 else '⚠️ BOŞ (ilk aramada doldurulacak)'} ({de_count} istasyon)")

# 5. train_cache
with sqlite3.connect(CACHE_DB) as conn:
    tc_count = conn.execute("SELECT COUNT(*) FROM train_cache").fetchone()[0]
print(f"  train_cache kayıt sayısı:      {tc_count} {'✅' if tc_count > 0 else '⚠️ BOŞ'}")

# 6. DE araması sonucu
de_result = results.get(("Berlin", "Munich"))
print(f"  Berlin→Munich araması:         {'✅ ' + str(len(de_result)) + ' sefer' if de_result is not None and not de_result.empty else '❌ SONUÇ YOK'}")

print("\n")

# Genel değerlendirme
issues = []
if not db_api_ok:
    issues.append("DB REST API'ye bağlanılamıyor (ağ/firewall kontrolü yapın)")
if not playwright_ok:
    issues.append("Playwright kurulu değil → 'uv add playwright && uv run playwright install chromium' çalıştırın")
if not roma_ok:
    issues.append("Roma Termini istasyon ID'si italya_istasyonlar.db'de eksik")
if de_result is not None and de_result.empty:
    issues.append("Berlin→Munich araması sonuç döndürmedi — API yanıtını inceleyin")

if issues:
    print("SORUNLAR:")
    for i, issue in enumerate(issues, 1):
        print(f"  {i}. {issue}")
else:
    print("✅ Tüm kontroller başarılı!")
print("=" * 60)

          TREN SİSTEMİ TANILAMA RAPORU
  DB REST API (Almanya):         ✅ Erişilebilir
  Playwright (İtalya için):       ✅ Yüklü
  İtalyan istasyon DB:           ✅ (206 istasyon)
  Roma Termini istasyon kaydı:   ✅ Mevcut
  Alman istasyon önbelleği:      ✅ (7 istasyon)
  train_cache kayıt sayısı:      4 ✅
  Berlin→Munich araması:         ✅ 5 sefer


✅ Tüm kontroller başarılı!
